# 65. Decoupled Bayesian Optimization 基礎

強度と導電率を別々に測定できる状況を考えます。部分観測を独立GPで学習し、次の設計点と測定項目を同時に選びます。理論は第33章を参照してください。

In [ ]:
import torch
import matplotlib.pyplot as plt
from botorch.fit import fit_gpytorch_mll
from botorch.models import ModelListGP, SingleTaskGP
from botorch.sampling.list_sampler import ListSampler
from botorch.sampling.normal import SobolQMCNormalSampler
from gpytorch.mlls import SumMarginalLogLikelihood
from robotorchan.acquisition.factory import create_acquisition
from robotorchan.optim.decoupled import optimize_acqf_decoupled
from robotorchan.workflow.cost_budget import CostBudget

torch.manual_seed(0)
torch.set_num_threads(1)
bounds = torch.tensor([[0.], [1.]], dtype=torch.double)
grid = torch.linspace(0, 1, 64, dtype=torch.double).unsqueeze(-1)
def truth(x):
    return torch.cat((1 - (x - .75)**2, 1 - (x - .25)**2), dim=-1)

X = [grid[[0, 20, 45]], grid[[0, 32, 63]]]
Y = [truth(x)[:, j:j+1] for j, x in enumerate(X)]
def fit_model():
    model = ModelListGP(*(SingleTaskGP(x, y) for x, y in zip(X, Y)))
    fit_gpytorch_mll(SumMarginalLogLikelihood(model.likelihood, model))
    model.eval()
    return model

model = fit_model()
print([x.shape for x in X], model.posterior(grid).mean.shape)

## 1. 部分観測の予測

各出力は異なる入力点で観測されています。潜在関数の事後平均と標準偏差を描き、観測のない領域の不確実性を確認します。

In [ ]:
def plot_model(model):
    with torch.no_grad():
        p = model.posterior(grid)
        mean, std = p.mean, p.variance.sqrt()
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    for j, ax in enumerate(axes):
        ax.plot(grid[:, 0], truth(grid)[:, j], '--', label='Truth')
        ax.plot(grid[:, 0], mean[:, j], label='Posterior mean')
        ax.fill_between(grid[:, 0], mean[:, j]-2*std[:, j], mean[:, j]+2*std[:, j], alpha=.2)
        ax.scatter(X[j][:, 0], Y[j][:, 0], label='Observed')
        ax.set_title(f'Output {j+1}')
        ax.legend()
    plt.show()

plot_model(model)

## 2. 評価マスクとEPIG

Trueの出力だけを測定します。EPIGは関心点の予測情報量を評価します。費用は出力1が1、出力2が3です。候補とマスク、費用補正後のランキング値を確認します。

In [ ]:
masks = torch.tensor([[True, False], [False, True], [True, True]])
costs = bounds.new_tensor([1., 3.])
def epig(model):
    return create_acquisition(
        "qDecoupledExpectedPredictiveInformationGain", model=model,
        target_X=grid, X_evaluation_mask=masks[:1],
        observation_noise=bounds.new_tensor([.01, .01]),
    )

def propose(acq, budget=None):
    return optimize_acqf_decoupled(
        acq, bounds, masks, output_costs=costs, budget=budget,
        optimizer="sobol", optimizer_options={"num_samples": 16}, seed=0,
    )

acq = epig(model)
with torch.no_grad():
    print("EPIG at x=0.5:", acq(bounds.new_tensor([[[.5]]])))
candidate, mask, score = propose(acq)
print(candidate, mask, score)

## 3. KGによる意思決定

KGは観測後に最良の設計を選び直す期待価値です。ここでは二つの出力を等重みで評価し、有限の意思決定集合gridを使います。

In [ ]:
def kg(model):
    return create_acquisition(
        "qDecoupledKnowledgeGradient", model=model, decision_X=grid,
        weights=bounds.new_tensor([.5, .5]),
        sampler=ListSampler(*(SobolQMCNormalSampler(torch.Size([4]), seed=j) for j in range(2))),
        X_evaluation_mask=masks[:1],
    )

candidate, mask, score = propose(kg(model))
print(candidate, mask, score)

## 4. HVKGによる多目的候補提案

HVKGは固定重みの代わりにHypervolumeを使います。one-shotの補助点は実験候補ではありません。ここでは小さなfantasy数で候補提案を確認します。

In [ ]:
hvkg = create_acquisition(
    "qHypervolumeKnowledgeGradient", model=model,
    ref_point=bounds.new_tensor([0., 0.]), num_fantasies=2,
    num_pareto=2, X_evaluation_mask=masks[:1],
)
candidate, mask, score = optimize_acqf_decoupled(
    hvkg, bounds, masks, num_restarts=2, raw_samples=8,
    options={"maxiter": 20},
)
print(candidate.shape, mask, score)

## 5. 部分観測の反復

初期観測費用も含め、総予算16でEPIGを使います。真値関数は実験装置の代わりです。選ばれた出力だけを学習データに追加し、再学習します。

In [ ]:
budget = CostBudget(16)
budget.record(sum(len(x)*float(c) for x, c in zip(X, costs)))
history = []
while budget.can_afford(float(costs.min())):
    candidate, mask, score = propose(epig(model), budget)
    charge = float((mask.to(costs) @ costs).sum())
    budget.record(charge)
    values = truth(candidate)
    for j in range(2):
        if mask[0, j]:
            X[j] = torch.cat((X[j], candidate))
            Y[j] = torch.cat((Y[j], values[:, j:j+1]))
    model = fit_model()
    history.append((budget.spent, *[len(x) for x in X]))
    print(candidate.tolist(), mask.tolist(), budget.spent)
plot_model(model)
h = torch.tensor(history)
plt.step(h[:, 0], h[:, 1], where='post', label='Output 1')
plt.step(h[:, 0], h[:, 2], where='post', label='Output 2')
plt.xlabel('Cumulative cost')
plt.ylabel('Observation count')
plt.legend()
plt.show()

## まとめ

部分観測では入力点と評価対象が別々の選択変数です。EPIGとKGでは価値の定義が異なり、HVKGではPareto集合を扱います。少数fantasyの例は使い方を学ぶためのもので、性能比較には共通予算と複数seedが必要です。